# Task 10 — CatBoost Learning-to-Rank

Read-only анализ опубликованного Task 10 artifact. Notebook не строит pools, не обучает модели и не изменяет результаты. Selection использует только `rolling_1 -> rolling_2` и `rolling_2 -> rolling_3`; canonical открывается после freeze единственного LTR winner.

`QueryCrossEntropy` исключён до full run: CatBoost 1.2.10 GPU ограничивает query size значением 256, тогда как bounded smoke зафиксировал train group до 273 и eval group до 709 candidates.

In [ ]:
import json
import sys
from pathlib import Path

import polars as pl
from IPython.display import display

repository_root = Path("..").resolve()
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from rankers import CatBoostRankerModel
from scripts.run_catboost_ltr import verify_ltr_artifact

artifact = repository_root / "artifacts/task10_catboost_ltr_v2"
if not artifact.is_dir():
    raise FileNotFoundError("Run Task 10 and verify its published artifact first")

config = json.loads((artifact / "config.json").read_text())
metrics = json.loads((artifact / "metrics.json").read_text())
stage_results = json.loads((artifact / "selection/stage_results.json").read_text())["stages"]
leaderboard = pl.read_parquet(artifact / "selection/leaderboard.parquet")

In [ ]:
summary = pl.DataFrame({
    "winner_config_id": [metrics["winner_config_id"]],
    "rolling_mean_p20_labeled": [metrics["selection"]["mean_precision_at_20_labeled_users"]],
    "rolling_min_p20_labeled": [metrics["selection"]["min_precision_at_20_labeled_users"]],
    "canonical_p20_all": [metrics["precision_at_20_all_targets"]],
    "canonical_p20_labeled": [metrics["precision_at_20_labeled_users"]],
    "canonical_hits": [metrics["final_hits"]],
    "tree_count": [metrics["tree_count"]],
    "runtime_hours": [metrics["runtime_seconds"] / 3600],
    "peak_memory_gib": [metrics["peak_memory_mb"] / 1024],
})
display(summary)

In [ ]:
display(
    leaderboard.sort(
        ["mean_precision_at_20_labeled_users", "min_precision_at_20_labeled_users", "fold_spread_precision_at_20_labeled_users"],
        descending=[True, True, False],
    )
)
display(pl.DataFrame([
    {
        "stage_id": stage["stage_id"],
        "incumbent_config_id": stage["incumbent_config_id"],
        "evaluated_config_count": len(stage["evaluated_config_ids"]),
        "mean_p20_labeled": stage["winner"]["mean_precision_at_20_labeled_users"],
        "min_p20_labeled": stage["winner"]["min_precision_at_20_labeled_users"],
    }
    for stage in stage_results
]))

In [ ]:
fold_rows = []
for result in metrics["selection"]["fold_results"]:
    fold_rows.append({
        "pair_id": result["pair_id"],
        "train_fold": result["train_fold"],
        "eval_fold": result["eval_fold"],
        "p20_all": result["precision_at_20_all_targets"],
        "p20_labeled": result["precision_at_20_labeled_users"],
        "hits": result["final_hits"],
        "trees": result["tree_count"],
        "runtime_minutes": result["runtime_seconds"] / 60,
    })
display(pl.DataFrame(fold_rows))

In [ ]:
comparisons = metrics["comparisons"]
comparison_rows = [
    {"model": "Task 10 LTR", "p20_all": metrics["precision_at_20_all_targets"], "p20_labeled": metrics["precision_at_20_labeled_users"], "hits": metrics["final_hits"]},
    {"model": "Task 09 pointwise s12", "p20_all": comparisons["task09_s12_canonical"]["precision_at_20_all_targets"], "p20_labeled": comparisons["task09_s12_canonical"]["precision_at_20_labeled_users"], "hits": comparisons["task09_s12_canonical"]["final_hits"]},
    {"model": "Task 08 pointwise", "p20_all": comparisons["task08"]["precision_at_20_all_targets"], "p20_labeled": comparisons["task08"]["precision_at_20_labeled_users"], "hits": comparisons["task08"]["final_hits"]},
    {"model": "Task 06 RRF full", "p20_all": comparisons["rrf_full"]["precision_at_20_all_targets"], "p20_labeled": comparisons["rrf_full"]["precision_at_20_labeled_users"], "hits": comparisons["rrf_full"]["final_hits"]},
]
display(pl.DataFrame(comparison_rows))

In [ ]:
pool_rows = []
for pair_id, manifest in config["pool_manifests"].items():
    pool_rows.append({
        "pair_id": pair_id,
        "train_rows": manifest["train_rows"],
        "train_groups": manifest["train_groups"],
        "train_zero_positive_groups": manifest["train_diagnostics"]["zero_positive_groups"],
        "eval_rows": manifest["eval_rows"],
        "eval_groups": manifest["eval_groups"],
        "borders_sha256": manifest["quantization"]["source_borders_sha256"],
    })
display(pl.DataFrame(pool_rows))
display(pl.read_parquet(artifact / "feature_importance.parquet").head(30))

In [ ]:
model = CatBoostRankerModel.from_artifact(artifact / "model")
assert model.tree_count == metrics["tree_count"]
assert metrics["canonical_evaluated_config_count"] == 1
display(model.get_config())

# Полная read-only проверка checksums, group mapping, selection replay,
# canonical recommendations и deterministic portable inference.
verification = verify_ltr_artifact(artifact)
display(verification)